# 03 · Morphological

Computes street features and plot area for each establishment.

- **Street features:** downloads the street network via OSMnx, matches each shop to its nearest street edge, extracts `highway_type` (categorical) and `lanes` (numeric).
- **Plot area:** loads NYC PLUTO tax-lot data, matches each shop to its nearest lot via BallTree, extracts `lot_area_sqft`.

**Input:** `csv/00_base_data.csv`, PLUTO CSV
**Output:** `csv/03_morphological.csv` — `osm_id`, `highway_type`, `lanes`, `lot_area_sqft`

In [ ]:
import pandas as pd
import numpy as np
import osmnx as ox
from sklearn.neighbors import BallTree

df = pd.read_csv("csv/00_base_data.csv")
print(f"pandas {pd.__version__} · osmnx {ox.__version__}")
print(f"Loaded {len(df)} records")

# PLUTO data path
PLUTO_CSV = "../ramy/NYC_pluto_25v4_csv/pluto_25v4.csv"

# Study area params (from 01_identifiers config)
ORIGIN_LAT = 40.7589
ORIGIN_LON = -73.9851
NETWORK_DIST = 1500  # meters for street network download

In [ ]:
# ── Street Features (highway type + lanes) via OSMnx ──

print("Downloading street network...")
G = ox.graph_from_point((ORIGIN_LAT, ORIGIN_LON), dist=NETWORK_DIST, network_type="drive")
print(f"  {len(ox.graph_to_gdfs(G, nodes=False))} street segments loaded")

def nearest_street_features(lat, lon):
    try:
        nearest = ox.nearest_edges(G, lon, lat)
        edge_data = G.edges[nearest]
        highway = edge_data.get("highway", None)
        lanes   = edge_data.get("lanes", None)
        if isinstance(highway, list):
            highway = highway[0]
        if isinstance(lanes, list):
            lanes = lanes[0]
        return pd.Series({
            "highway_type": str(highway) if highway else None,
            "lanes"       : float(lanes) if lanes else None,
        })
    except Exception:
        return pd.Series({"highway_type": None, "lanes": None})

df[["highway_type", "lanes"]] = df.apply(
    lambda row: nearest_street_features(row["lat"], row["lon"]), axis=1
)

print(f"\nhighway_type fill : {df['highway_type'].notna().sum()}/{len(df)}")
print(f"lanes fill        : {df['lanes'].notna().sum()}/{len(df)}")
print(f"\nhighway_type distribution:")
print(df["highway_type"].value_counts())

In [ ]:
# ── Plot Area (lot_area_sqft) via PLUTO + BallTree ────

print("Loading PLUTO data...")
pluto = pd.read_csv(PLUTO_CSV, low_memory=False)
pluto_mn = pluto[pluto["borough"] == "MN"].dropna(subset=["latitude", "longitude"]).copy()
print(f"  {len(pluto_mn)} Manhattan lots loaded")

# Build spatial index on PLUTO lots
pluto_coords = np.radians(pluto_mn[["latitude", "longitude"]].values)
tree = BallTree(pluto_coords, metric="haversine")

# Query nearest lot for each shop
shop_coords = np.radians(df[["lat", "lon"]].values)
distances, indices = tree.query(shop_coords, k=1)

# Attach lotarea from nearest lot
df["lot_area_sqft"] = pluto_mn.iloc[indices.flatten()]["lotarea"].values

print(f"\nlot_area_sqft fill : {df['lot_area_sqft'].notna().sum()}/{len(df)}")
print(f"  Mean lot area : {df['lot_area_sqft'].mean():.0f} sqft")
print(f"  Min  lot area : {df['lot_area_sqft'].min():.0f} sqft")
print(f"  Max  lot area : {df['lot_area_sqft'].max():.0f} sqft")

In [ ]:
df_out = df[["osm_id", "highway_type", "lanes", "lot_area_sqft"]]
df_out.to_csv("csv/03_morphological.csv", index=False, encoding="utf-8")
print(f"Saved {len(df_out)} records to csv/03_morphological.csv")
print(df_out.describe(include="all").round(1))